In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
import copy
import gc
import warnings

warnings.filterwarnings('ignore')

## 1. Configuration & Reproducibility

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BLOCK_SIZE = 4
OPTIMAL_P = 0.05

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

## 2. Metrics & Data Utilities

In [3]:
def calculate_model_size_mb(model, bitwidth=2, protect_ratio=0.05, block_size=4):
    total_bytes = 0
    for name, param in model.named_parameters():
        num_params = param.numel()
        if 'weight' in name and isinstance(model.get_submodule(name.rsplit('.', 1)[0]), nn.Linear):
            if any(k in name for k in ["classifier", "score", "pooler"]):
                total_bytes += num_params * 2
            else:
                if bitwidth == 16:
                    total_bytes += num_params * 2
                else:
                    out_features, in_features = param.shape
                    protected_cols = max(1, int(in_features * protect_ratio)) if protect_ratio > 0 else 0
                    quantized_cols = in_features - protected_cols

                    outlier_params = out_features * protected_cols
                    quantized_params = out_features * quantized_cols

                    # Block alignment padding computation
                    remainder = quantized_cols % (block_size * block_size)
                    pad_len = ((block_size * block_size) - remainder) if remainder != 0 else 0
                    padded_quantized_params = out_features * (quantized_cols + pad_len)

                    num_blocks = padded_quantized_params / (block_size * block_size)
                    scales_bytes = num_blocks * 2

                    total_bytes += (quantized_params * (bitwidth / 8.0)) + (outlier_params * 2) + scales_bytes
        else:
            total_bytes += num_params * 2

    return total_bytes / (1024 ** 2)

def load_evaluation_data(tokenizer, dataset_id, task_type):
    dataset = load_dataset(dataset_id, split="validation")
    if task_type == "mrpc":
        texts = [f"{t1} {tokenizer.sep_token} {t2}" for t1, t2 in zip(dataset["text1"], dataset["text2"])]
    else:
        texts = dataset["text"]
    labels = torch.tensor(dataset["label"]).to(DEVICE)
    return texts, labels

def evaluate_and_check_collapse(model, tokenizer, texts, labels, batch_size=32):
    model.eval()
    all_preds = []
    correct = 0
    total = len(texts)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=128).to(DEVICE)
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            correct += (preds == batch_labels).sum().item()

    accuracy = 100 * correct / total
    unique, counts = torch.unique(torch.tensor(all_preds), return_counts=True)
    dist = dict(zip(unique.tolist(), counts.tolist()))

    return accuracy, dist

## 3. Quantization Engines & Neuron-Safe Wrapper
Contains the baseline Absolute Maximum (AbsMax) quantization and the proposed S-DPQ algorithm.

In [4]:
def classic_rtn_core(w):
    w_fp32 = w.float()
    w_reshaped = w_fp32.view(-1, BLOCK_SIZE, BLOCK_SIZE)
    max_w = w_reshaped.abs().amax(dim=(1,2), keepdim=True)
    scale = torch.where(max_w == 0, torch.tensor(1e-5, device=DEVICE), max_w)
    q_res = torch.round(w_reshaped / scale) * scale
    return q_res.view_as(w).to(w.dtype)

def s_dpq_core(w):
    w_fp32 = w.float()
    w_reshaped = w_fp32.view(-1, BLOCK_SIZE, BLOCK_SIZE)
    B = w_reshaped.shape[0]

    max_w = w_reshaped.abs().amax(dim=(1,2), keepdim=True)
    scale_fallback = torch.where(max_w == 0, torch.tensor(1e-5, device=DEVICE), max_w)
    best_w_quant = torch.round(w_reshaped / scale_fallback) * scale_fallback

    norm_w = torch.linalg.norm(w_reshaped, dim=(1,2))
    best_cos = torch.full((B,), -float('inf'), device=DEVICE)

    for factor in [0.2, 0.4, 0.6]:
        th = max_w * factor
        q = torch.zeros_like(w_reshaped)
        q[w_reshaped > th] = 1.0
        q[w_reshaped < -th] = -1.0

        norm_q = torch.linalg.norm(q, dim=(1,2))
        valid = (norm_w > 1e-5)

        scale_energy = torch.zeros(B, device=DEVICE)
        safe_norm_q = torch.where(norm_q == 0, torch.tensor(1.0, device=DEVICE), norm_q)
        scale_energy[valid] = norm_w[valid] / safe_norm_q[valid]

        w_q = q * scale_energy.view(B, 1, 1)

        norm_w_q = torch.linalg.norm(w_q, dim=(1,2))
        safe_norm_w_q = torch.where(norm_w_q == 0, torch.tensor(1.0, device=DEVICE), norm_w_q)
        cos_sim = (w_reshaped * w_q).sum(dim=(1,2)) / (norm_w * safe_norm_w_q + 1e-6)

        improve = valid & (cos_sim > best_cos)
        best_cos = torch.where(improve, cos_sim, best_cos)
        best_w_quant = torch.where(improve.view(B, 1, 1), w_q, best_w_quant)

    return best_w_quant.view_as(w).to(w.dtype)

def outlier_aware_wrapper_columns_fixed(weight_tensor, quant_func, protect_ratio):
    with torch.no_grad():
        W = weight_tensor.clone()
        out_features, in_features = W.shape

        if protect_ratio > 0:
            col_norms = torch.linalg.norm(W.float(), dim=0)
            k = max(1, int(in_features * protect_ratio))
            threshold_val = torch.topk(col_norms, k)[0][-1]
            is_outlier_col = col_norms >= threshold_val
        else:
            is_outlier_col = torch.zeros(in_features, dtype=torch.bool, device=DEVICE)

        W_safe = W[:, ~is_outlier_col]
        safe_cols = W_safe.shape[1]

        # Pad to the next multiple of B^2=16 for contiguous memory reshaping
        remainder = safe_cols % 16
        pad_len = (16 - remainder) if remainder != 0 else 0
        if pad_len > 0:
            padding = torch.zeros((out_features, pad_len), dtype=W.dtype, device=DEVICE)
            W_safe_padded = torch.cat([W_safe, padding], dim=1)
        else:
            W_safe_padded = W_safe

        w_blocked = W_safe_padded.view(-1, BLOCK_SIZE, BLOCK_SIZE)
        w_q_blocked = quant_func(w_blocked)
        W_q_padded = w_q_blocked.view(out_features, -1)

        if pad_len > 0:
            W_q_safe = W_q_padded[:, :-pad_len]
        else:
            W_q_safe = W_q_padded

        W_final = W.clone()
        if (~is_outlier_col).any():
            W_final[:, ~is_outlier_col] = W_q_safe.to(W.dtype)

        return W_final

def quantize_network(model, quant_func, protect_ratio=0.05):
    model_q = copy.deepcopy(model)
    for name, module in model_q.named_modules():
        if isinstance(module, nn.Linear) and not any(k in name for k in ["classifier", "score", "pooler"]):
            module.weight.data = outlier_aware_wrapper_columns_fixed(module.weight.data, quant_func, protect_ratio)
    return model_q

## 4. Experiment Pipelines
Functions to run the main benchmark (Table 1) and the ablation study (Table 2).

In [5]:
def run_main_experiment(exp_name, model_id, dataset_id, task_type, batch_size):
    print(f"\n--- {exp_name} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(DEVICE)
    texts, labels = load_evaluation_data(tokenizer, dataset_id, task_type)

    acc_orig, dist_orig = evaluate_and_check_collapse(model, tokenizer, texts, labels, batch_size)
    size_orig = calculate_model_size_mb(model, bitwidth=16, protect_ratio=OPTIMAL_P)

    model_classic = quantize_network(model, classic_rtn_core, protect_ratio=OPTIMAL_P)
    acc_classic, dist_classic = evaluate_and_check_collapse(model_classic, tokenizer, texts, labels, batch_size)
    size_q = calculate_model_size_mb(model_classic, bitwidth=2, protect_ratio=OPTIMAL_P)

    del model_classic
    torch.cuda.empty_cache()
    gc.collect()

    model_sdpq = quantize_network(model, s_dpq_core, protect_ratio=OPTIMAL_P)
    acc_sdpq, dist_sdpq = evaluate_and_check_collapse(model_sdpq, tokenizer, texts, labels, batch_size)

    del model_sdpq
    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Original (16-bit)   | Acc: {acc_orig:5.2f}% | Size: {size_orig:6.2f} MB | Dist: {dist_orig}")
    print(f"AbsMax RTN (2-bit)  | Acc: {acc_classic:5.2f}% | Size: {size_q:6.2f} MB | Dist: {dist_classic}")
    print(f"S-DPQ (2-bit)       | Acc: {acc_sdpq:5.2f}% | Size: {size_q:6.2f} MB | Dist: {dist_sdpq}")

def run_ablation_study(exp_name, model_id, dataset_id, task_type, batch_size):
    print(f"\n--- Ablation Study: {exp_name} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(DEVICE)
    texts, labels = load_evaluation_data(tokenizer, dataset_id, task_type)

    configs = [
        {"name": "No Protection (p=0.00)", "protect": 0.00},
        {"name": "Tested Config (p=0.05)", "protect": 0.05},
        {"name": "Over-Protection (p=0.10)", "protect": 0.10},
    ]

    for cfg in configs:
        model_q = quantize_network(model, s_dpq_core, protect_ratio=cfg["protect"])
        acc, dist = evaluate_and_check_collapse(model_q, tokenizer, texts, labels, batch_size=batch_size)
        print(f"{cfg['name']:<25} | Acc: {acc:5.2f}% | Dist: {dist}")

        del model_q
        torch.cuda.empty_cache()
        gc.collect()

    del model
    torch.cuda.empty_cache()
    gc.collect()

## 5. Execution (Reproducing Paper Results)

In [7]:
# ==========================================
# Reproducing Table 1: Generalization and Boundary Failures
# ==========================================
run_main_experiment("DistilBERT (66M) on SST-2", "distilbert-base-uncased-finetuned-sst-2-english", "SetFit/sst2", "sst2", 64)
run_main_experiment("DistilBERT (66M) on MRPC", "textattack/distilbert-base-uncased-MRPC", "SetFit/mrpc", "mrpc", 64)

run_main_experiment("RoBERTa-Large (355M) on SST-2", "siebert/sentiment-roberta-large-english", "SetFit/sst2", "sst2", 16)
run_main_experiment("RoBERTa-Large (355M) on MRPC", "howey/roberta-large-mrpc", "SetFit/mrpc", "mrpc", 16)

# ==========================================
# Reproducing Table 2: Ablation Study
# ==========================================
run_ablation_study("RoBERTa-Large (355M) on MRPC", "howey/roberta-large-mrpc", "SetFit/mrpc", "mrpc", 16)


--- DistilBERT (66M) on SST-2 ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Original (16-bit)   | Acc: 90.94% | Size: 127.71 MB | Dist: {0: 413, 1: 459}
AbsMax RTN (2-bit)  | Acc: 82.57% | Size:  65.19 MB | Dist: {0: 450, 1: 422}
S-DPQ (2-bit)       | Acc: 87.84% | Size:  65.19 MB | Dist: {0: 396, 1: 476}

--- DistilBERT (66M) on MRPC ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Repo card metadata block was not found. Setting CardData to empty.


Original (16-bit)   | Acc: 87.75% | Size: 677.80 MB | Dist: {0: 133, 1: 275}
AbsMax RTN (2-bit)  | Acc: 60.05% | Size: 233.21 MB | Dist: {0: 92, 1: 316}
S-DPQ (2-bit)       | Acc: 68.38% | Size: 233.21 MB | Dist: {1: 408}

--- Ablation Study: RoBERTa-Large (355M) on MRPC ---


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


No Protection (p=0.00)    | Acc: 31.62% | Dist: {0: 408}
Tested Config (p=0.05)    | Acc: 68.38% | Dist: {1: 408}
Over-Protection (p=0.10)  | Acc: 31.62% | Dist: {0: 408}
